# Nightly Report: IQ, AOS, Thermal, Aberrations and Degrees of Freedom

Owner: **Guillem Megias** <br>
Last Verified to Run: **2025-07-07** <br>

In [ ]:
# Times Square Parameters
day_obs = 20260111
seq_min = 0
seq_max = 1500

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.dates import DateFormatter
from matplotlib.patches import Rectangle, Patch
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
import galsim
import numpy as np
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

def getPsfGradPerZernike(
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
    jmax: int = 22,
) -> np.ndarray:
    """Get the gradient of the PSF FWHM with respect to each Zernike.

    This function takes no positional arguments. All parameters must be passed
    by name (see the list of parameters below).

    Parameters
    ----------
    diameter : float, optional
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float, optional
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int, optional
        The minimum Noll index, inclusive. Must be >= 0. (the default is 4)
    jmax : int, optional
        The max Zernike Noll index, inclusive. Must be >= jmin.
        (the default is 22.)

    Returns
    -------
    np.ndarray
        Gradient of the PSF FWHM with respect to the corresponding Zernike.
        Units are arcsec / micron.

    Raises
    ------
    ValueError
        If jmin is negative or jmax is less than jmin
    """
    # Check jmin and jmax
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")
    if jmax < jmin:
        raise ValueError("jmax must be greater than jmin.")

    # Calculate the conversion factors
    conversion_factors = np.zeros(jmax + 1)
    for i in range(jmin, jmax + 1):
        # Set coefficients for this Noll index: coefs = [0, 0, ..., 1]
        # Note the first coefficient is Noll index 0, which does not exist and
        # is therefore always ignored by galsim
        coefs = [0] * i + [1]

        # Create the Zernike polynomial with these coefficients
        R_outer = diameter / 2
        R_inner = R_outer * obscuration
        Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

        # We can calculate the size of the PSF from the RMS of the gradient of
        # the wavefront. The gradient of the wavefront perturbs photon paths.
        # The RMS quantifies the size of the collective perturbation.
        # If we expand the wavefront gradient in another series of Zernike
        # polynomials, we can exploit the orthonormality of the Zernikes to
        # calculate the RMS from the Zernike coefficients.
        rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)

        # Convert to arcsec per micron
        rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600

        # Convert rms -> fwhm
        fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt

        # Save this conversion factor
        conversion_factors[i] = fwhm_tilt

    return conversion_factors[jmin:]


def convertZernikesToPsfWidth(
    zernikes: np.ndarray,
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
) -> np.ndarray:
    """Convert Zernike amplitudes to quadrature contribution to the PSF FWHM.

    Parameters
    ----------
    zernikes : np.ndarray
        Zernike amplitudes (in microns), starting with Noll index `jmin`.
        Either a 1D array of zernike amplitudes, or a 2D array, where each row
        corresponds to a different set of amplitudes.
    diameter : float
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int
        The minimum Zernike Noll index, inclusive. Must be >= 0. The
        max Noll index is inferred from `jmin` and the length of `zernikes`.
        (the default is 4, which ignores piston, x & y offsets, and tilt.)

    Returns
    -------
    dFWHM: np.ndarray
        Quadrature contribution of each Zernike vector to the PSF FWHM
        (in arcseconds).

    Notes
    -----
    Converting Zernike amplitudes to their quadrature contributions to the PSF
    FWHM allows for easier physical interpretation of Zernike amplitudes and
    the performance of the AOS system.

    For example, image we have a true set of zernikes, [Z4, Z5, Z6], such that
    ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.2, 0.3] arcsecs.
    These Zernike perturbations increase the PSF FWHM by
    sqrt[(0.1)^2 + (-0.2)^2 + (0.3)^2] ~ 0.37 arcsecs.

    If the AOS perfectly corrects for these perturbations, the PSF FWHM will
    not increase in size. However, imagine the AOS estimates zernikes, such
    that ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.3, 0.4] arcsecs.
    These estimated Zernikes, do not exactly match the true Zernikes above.
    Therefore, the post-correction PSF will still be degraded with respect to
    the optimal PSF. In particular, the PSF FWHM will be increased by
    sqrt[(0.1 - 0.1)^2 + (-0.2 - (-0.3))^2 + (0.3 - 0.4)^2] ~ 0.14 arcsecs.

    This conversion depends on a linear approximation that begins to break down
    for RSS(dFWHM) > 0.20 arcsecs. Beyond this point, the approximation tends
    to overestimate the PSF degradation. In other words, if
    sqrt(sum( dFWHM^2 )) > 0.20 arcsec, it is likely that dFWHM is
    over-estimated. However, the point beyond which this breakdown begins
    (and whether the approximation over- or under-estimates dFWHM) can change,
    depending on which Zernikes have large amplitudes. In general, if you have
    large Zernike amplitudes, proceed with caution!
    Note that if the amplitudes Z_est and Z_true are large, this is okay, as
    long as |Z_est - Z_true| is small.

    For a notebook demonstrating where the approximation breaks down:
    https://gist.github.com/jfcrenshaw/24056516cfa3ce0237e39507674a43e1

    Raises
    ------
    ValueError
        If jmin is negative
    """
    # Check jmin
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")

    # Calculate jmax from jmin and the length of the zernike array
    jmax = jmin + np.array(zernikes).shape[-1] - 1

    # Calculate the conversion factors for each zernike
    conversion_factors = getPsfGradPerZernike(
        jmin=jmin,
        jmax=jmax,
        diameter=diameter,
        obscuration=obscuration,
    )

    # Convert the Zernike amplitudes from microns to their quadrature
    # contribution to the PSF FWHM
    dFWHM = conversion_factors * zernikes

    return dFWHM

band_colors = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}
    
def annotate_bands(data: pd.DataFrame, ax: plt.Axes) -> None:
    """Anotate bottom of plot with band colors

    Parameters
    ----------
    data : pd.DataFrame
        Table of data that is being plotted
    ax : plt.Axes
        Axis on which to add annotations
    """
    # Get the bands
    bands = data['band']

    # Get the axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Plot band bars at bottom of plot
    max_seq = data.index[-1]
    for i, band in enumerate(bands[:-1]):
        ax.plot(
            data.index[i : i + 2],
            2 * [ylim[0]],
            c=band_colors[band[0]],
            lw=7,
        )
    # Do something special for last one
    ax.plot(
        [data.index[-1], data.index[-1]+1],
        2 * [ylim[0]],
        c=band_colors[bands.iloc[-1][0]],
        lw=7,
    )

    # Restore the axis limits
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)

    # Add unique bands to legend
    unique_bands = data['band'].str[0].unique()  # Get first character of each band
    
    for band_char in sorted(unique_bands):
        ax.scatter([], [], 
                  c=band_colors[band_char], 
                  s=100, 
                  marker='s', 
                  label=f'{band_char}')

def find_blocks(data: pd.DataFrame) -> pd.DataFrame:
    """Find blocks active during the night

    Parameters
    ----------
    data : pd.DataFrame
        Table of data that is being plotted
    Returns
    -------
    changes: pd.DataFrame
        dataframe containing the sequence numbers where the block changed
        and the name of the new block
    """
    block_data = data[['seq', 'block']].sort_values(by='seq')
    change_mask = block_data['block'] != block_data['block'].shift()
    block_data['block'] = block_data['block'].str.replace('BLOCK-', '', regex=False)
    blocks = pd.DataFrame({
        'seq': block_data.loc[change_mask, 'seq'],
        'block_after': block_data['block'].loc[change_mask]
    })
    return blocks

async def find_faults(client, table):
    """Find faults during the night

    Parameters
    ----------
    client: EFD client
    table : pd.DataFrame
        Table of data that is being plotted
    Returns
    -------
    table: pd.DataFrame
        incoming dataframe with the fault events added
    """
    table_time = pd.to_datetime(table['time'], format='ISO8601', utc=True)
    topics = ['MTMount', 'MTAOS', 'MTHexapod', 'MTCamera']
    table['faults'] = [None for i in range(len(table))]
    start = Time(table['obs_start'].iloc[0])
    end = Time(table['obs_end'].iloc[-1])
    for topic in topics:
        efd_data = await client.select_time_series(f"lsst.sal.{topic}.logevent_summaryState", \
                                                ['summaryState'], \
                                                 start, end)
        if len(efd_data) == 0:
            continue
        faults = efd_data[efd_data['summaryState'] == 3]
        for i in range(len(faults)):
            fault_time = pd.to_datetime(faults.index[i], format='utc')
            closest_row = table.iloc[(table_time - fault_time).abs().argmin()]
            table.loc[table['seq'] == closest_row['seq'], 'faults'] = topic
    return table


In [ ]:
from lsst.ts.xml.tables.m1m3 import *
from lsst.ts.m1m3.utils import *


async def get_m1m3_gradients(client, data):
    """Get the M1M3 thermal gradients

    Parameters
    ----------
    client: EFD client
    data : pd.DataFrame
        Table of minimal data
    Returns
    -------
    data: pd.DataFrame
        incoming dataframe with the m1m3 temperature gradients added
    """
    data_times = pd.to_datetime(data['obs_start'], format='ISO8601', utc=True)
    sorted_data_times = data_times.sort_values()
    start = Time(sorted_data_times.iloc[0])
    end = Time(sorted_data_times.iloc[-1])
    data_times = data_times.astype('int64')
    thermocouples = ThermocoupleAnalysis(client)
    await thermocouples.load(start, end, time_bin=30)
    gradients = thermocouples.xyz_r_gradients
    grad_times = pd.to_datetime(gradients.index, format='ISO8601', utc=True).astype('int64')
    t0 = grad_times[0]
    grad_times -= t0
    grad_times /=1E9
    data_times -= t0
    data_times /= 1E9
    names = ['x_gradient', 'y_gradient', 'z_gradient', 'radial_gradient']
    for name in names:
        values = gradients[name].values
        val_series = pd.Series(values)
        val_interpolated = val_series.interpolate()
        data[name] = np.interp(data_times, grad_times, val_interpolated)
    return data


In [ ]:
from matplotlib.patches import Patch
from matplotlib import cm

def plot_m1m3_gradients(data, ax):
    """Plot the M1M3 thermal gradients

    Parameters
    ----------
    data : pd.DataFrame
        Table of data to plot
    ax : matplotlib.axes._axes.Axes
        axes object on which to plot the data
    Returns
    -------
    """

    # --- Palettes: same hue family within groups, distinct shades per line ---
    blues   = plt.get_cmap('Blues')
    oranges = plt.get_cmap('Oranges')
    
    # Line colors (distinct, but clearly grouped)
    c_x = blues(0.75)     # darker blue
    c_y = blues(0.55)     # lighter blue
    c_r = oranges(0.75)   # darker orange
    c_z = oranges(0.55)   # lighter orange
    
    # Band colors (very light tint of the group hue)
    band_xy = blues(0.25)
    band_rz = oranges(0.25)
    
    # --- Shaded operational bands (put behind data) ---
    ax.axhspan(-0.4, 0.4, facecolor=band_xy, alpha=0.3, zorder=0)
    ax.axhspan(-0.1, 0.1, facecolor=band_rz, alpha=0.3, zorder=0)
    ax.axhline(0.4,  color=blues(0.65), linestyle="-", linewidth=1, alpha=0.5)
    ax.axhline(-0.4, color=blues(0.65), linestyle="-", linewidth=1, alpha=0.5)
    ax.axhline(0.1,  color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.5)
    ax.axhline(-0.1, color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.5)
    
    # --- Data lines: distinct per series with styles/markers ---
    ax.scatter(data['seq'],
        data['x_gradient'] * 8.4,
        label="X (×8.4)", color=c_x, linewidth=2.0, linestyle="-",
        s=0.5, zorder=3
    )
    ax.scatter(data['seq'],
        data['y_gradient'] * 8.4,
        label="Y (×8.4)", color=c_y, linewidth=2.0, linestyle="--",
        s=0.5, zorder=3
    )
    ax.scatter(data['seq'],
        data['radial_gradient'] * 3.7,
        label="Radial (×4.2)", color=c_r, linewidth=2.0, linestyle="-",
        s=0.5, zorder=4
    )
    ax.scatter(data['seq'],
        data['z_gradient'],
        label="Z", color=c_z, linewidth=2.0, linestyle="--",
        s=0.5, zorder=4
    )
    
    
    # --- Legends: one for data lines, one for bands (with matching colors) ---
    data_leg = ax.legend(bbox_to_anchor=(0.0, 1.25), loc='upper left',
                frameon=True, ncol=2, markerscale=4)
    ax.add_artist(data_leg)
    
    band_handles = [
        Patch(facecolor=band_xy, alpha=0.6, label="X/Y limit band (±0.4)"),
        Patch(facecolor=band_rz, alpha=0.6, label="Radial/Z limit band (±0.1)"),
    ]
    ax.legend(handles=band_handles,
              loc="upper right", frameon=True, bbox_to_anchor=(1.0, 1.25))
    
    # --- Labels, grid, cosmetics ---
    
    ax.set_ylim(-0.6, 0.6)
    return


In [ ]:
import logging

from astropy.time import Time, TimeDelta
from lsst.obs.lsst import LsstCam
from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getMostRecentRowWithDataBefore,
    makeEfdClient,
)

from tqdm import tqdm as tqdm

__all__ = ["AOSDatabase"]


class AOSDatabase:
    table: pd.DataFrame

    def __init__(
        self,
        day_obs: int = 20250415,
        seq_min: int = 1,
        seq_max: int = 9999,
        consdb_url: str = "http://consdb-pq.consdb:8080/consdb",
    ) -> None:
        """Create fetcher.

        Parameters
        ----------
        seq_max : int, optional
            Maximum sequence number to fetch. Default is 9999.
        consdb_url : str, optional
            URL to create ConsDB client.
            The default is "http://consdb-pq.consdb:8080/consdb".
        """
        self.log = logging.getLogger(__name__)

        self.efd_client = makeEfdClient()
        self.cdb_client = ConsDbClient(consdb_url)

        self.det_order = list([191, 195, 199, 203])
        camera = LsstCam().getCamera()
        self.detector_names = [
            camera.get(det_id).getName() for det_id in self.det_order
        ]

        self.day_obs = day_obs
        self.seq_max = seq_max
        self.seq_min = seq_min
        self.table = pd.DataFrame()

        self.time_window = TimeDelta(0.2, format="sec")
        self.temp_time_window = TimeDelta(0.2, format="sec")

    async def create(self, simplified=False):
        self.table = await self._fetch(
            self.day_obs, self.seq_min, self.seq_max, simplified
        )

    async def update(self, simplified: bool = False) -> None:
        """Update the database by grabbing more recent exposures.
        This will only grab new sequences, not re-fetch existing ones.
        """
        # First grab new sequences
        seq_min = self.table["seq"].max() + 1
        updated_table = await self._fetch(
            self.day_obs, seq_min, self.seq_max, simplified=simplified
        )
        self.table = pd.concat(
            [self.table, updated_table],
            ignore_index=True,
        )
    
    async def _fetch(
        self, day_obs: int, seq_min: int, seq_max: int, simplified: bool = False
    ) -> pd.DataFrame:
        query = f"""
            SELECT
            e.air_temp AS air_temp,
            e.airmass AS airmass,
            e.dimm_seeing AS dimm,
            e.altitude AS elevation,
            e.azimuth AS azimuth,
            e.exposure_id AS visit_id,
            e.physical_filter as band,
            e.day_obs AS day_obs,
            e.exp_midpt AS time,
            e.dimm_seeing AS seeing,
            e.seq_num AS seq,
            e.science_program AS block,
            ccdvisit1_quicklook.psf_sigma,
            ccdvisit1_quicklook.z4,
            ccdvisit1_quicklook.z5,
            ccdvisit1_quicklook.z6,
            ccdvisit1_quicklook.z7,
            ccdvisit1_quicklook.z8,
            ccdvisit1_quicklook.z9,
            ccdvisit1_quicklook.z10,
            ccdvisit1_quicklook.z11,
            ccdvisit1_quicklook.z12,
            ccdvisit1_quicklook.z13,
            ccdvisit1_quicklook.z14,
            ccdvisit1_quicklook.z15,
            ccdvisit1_quicklook.z16,
            ccdvisit1_quicklook.z17,
            ccdvisit1_quicklook.z18,
            ccdvisit1_quicklook.z19,
            ccdvisit1_quicklook.z20,
            ccdvisit1_quicklook.z21,
            ccdvisit1_quicklook.z22,
            ccdvisit1_quicklook.z23,
            ccdvisit1_quicklook.z24,
            ccdvisit1_quicklook.z25,
            ccdvisit1_quicklook.z26,
            ccdvisit1_quicklook.z27,
            ccdvisit1_quicklook.z28,
            ccdvisit1.detector as detector,
            q.psf_sigma_median AS psf_fwhm,
            q.psf_sigma_min AS psf_fwhm_min,
            q.psf_sigma_max AS psf_fwhm_max,
            q.psf_area_median AS psf_area,
            q.psf_area_min AS psf_area_min,
            q.psf_area_max AS psf_area_max,
            q.aos_fwhm AS aos_fwhm,
            q.donut_blur_fwhm AS donut_blur_fwhm,
            q.physical_rotator_angle AS rotation_angle,
            e.obs_end,
            e.obs_start
            FROM
            cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
            cdb_lsstcam.ccdvisit1 AS ccdvisit1,
            cdb_lsstcam.visit1 AS visit1,
            cdb_lsstcam.visit1_quicklook AS q,
            cdb_lsstcam.exposure AS e
            WHERE
            ccdvisit1.detector IN (191, 192, 195, 196, 199, 200, 203, 204)
            AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
            AND ccdvisit1.visit_id = visit1.visit_id
            AND ccdvisit1.visit_id = q.visit_id
            AND ccdvisit1.visit_id = e.exposure_id
            AND (e.img_type = 'science' or e.img_type = 'acq' or e.img_type = 'engtest')
            AND e.day_obs = {day_obs}
            AND (e.seq_num BETWEEN {seq_min} AND {seq_max})
        """
        self.table = self.cdb_client.query(query).to_pandas()
        if len(self.table) == 0:
            #raise RuntimeError("ConsDB query is empty")
            return self.table

        # Correctly declare aos_fwhm and donut_blur_fwhm as float
        self.table['psf_fwhm'] = pd.to_numeric(self.table['psf_fwhm'])
        self.table['aos_fwhm'] = pd.to_numeric(self.table['aos_fwhm'])
        self.table['donut_blur_fwhm'] = pd.to_numeric(self.table['donut_blur_fwhm'])

        # Convert PSF sigma to FWHM
        sig2fwhm = 2 * np.sqrt(2 * np.log(2))
        pixel_scale = 0.2  # arcsec / pixel
        self.table["psf_fwhm"] = self.table["psf_fwhm"] * sig2fwhm * pixel_scale
        self.table["psf_fwhm_min"] = self.table["psf_fwhm_min"] * sig2fwhm * pixel_scale
        self.table["psf_fwhm_max"] = self.table["psf_fwhm_max"] * sig2fwhm * pixel_scale
        self.table['airmass'] = np.clip(self.table['airmass'], 1.0, 3.0)

        self.table["fwhm_zenith_500nm"] = [
            fwhm
            * getAirmassSeeingCorrection(airmass)
            * getBandpassSeeingCorrection(band)
            for fwhm, band, airmass in zip(
                self.table["psf_fwhm"], self.table["band"], self.table["airmass"]
            )
        ]

        zernike_columns = [f"z{i}" for i in range(4, 29)]
        self.table["zernikes"] = self.table[zernike_columns].apply(
            lambda row: np.array(row.fillna(0.0).values, dtype=float), axis=1
        )
        self.table["zernikes_fwhm"] = self.table["zernikes"].apply(
            convertZernikesToPsfWidth
        )
        
        # Get the data for the science CCDs for fwhm_05 and fwhm_95
        visits_query = f'''
        SELECT 
        ccdvisit1_quicklook.psf_sigma,
        ccdvisit1_quicklook.psf_ixx,
        ccdvisit1_quicklook.psf_iyy,
        ccdvisit1_quicklook.psf_ixy,
        ccdvisit1.detector,
        visit1.visit_id,
        visit1.seq_num AS seq,
        visit1.day_obs,
        visit1.airmass
        FROM
        cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
        cdb_lsstcam.ccdvisit1 AS ccdvisit1,
        cdb_lsstcam.visit1_quicklook AS visit1_quicklook,
        cdb_lsstcam.visit1 AS visit1 
        WHERE 
        ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
        AND ccdvisit1.visit_id = visit1.visit_id 
        AND visit1.visit_id = visit1_quicklook.visit_id
        AND ccdvisit1.detector NOT IN (168, 188, 123, 27, 0, 20, 65, 161)
        AND visit1.airmass > 0
        AND visit1.day_obs = {self.day_obs}
        AND (visit1.seq_num BETWEEN {self.seq_min} AND {self.seq_max})
        AND (visit1.img_type = 'science' or visit1.img_type = 'acq' or visit1.img_type = 'engtest')
        '''
        
        ccdvisits = self.cdb_client.query(visits_query).to_pandas()

        ccdvisits["psf_sigma"] = pd.to_numeric(ccdvisits["psf_sigma"], errors="coerce")
        ccdvisits["psf_ixx"]   = pd.to_numeric(ccdvisits["psf_ixx"], errors="coerce")
        ccdvisits["psf_iyy"]   = pd.to_numeric(ccdvisits["psf_iyy"], errors="coerce")
        ccdvisits["psf_ixy"]   = pd.to_numeric(ccdvisits["psf_ixy"], errors="coerce")

        ccdvisits["psf_fwhm"] = ccdvisits["psf_sigma"] * sig2fwhm * pixel_scale

        denom = ccdvisits["psf_ixx"] + ccdvisits["psf_iyy"]
        denom = denom.replace(0, np.nan)
        
        e1 = (ccdvisits["psf_ixx"] - ccdvisits["psf_iyy"]) / denom
        e2 = (2 * ccdvisits["psf_ixy"]) / denom
        
        ccdvisits["ellipticity"] = np.sqrt(e1.to_numpy()**2 + e2.to_numpy()**2)
        
        # --- Group by visit -------------------------------------------------------
        clean = ccdvisits.dropna(subset=["psf_fwhm", "ellipticity"])
        groups = clean.groupby("visit_id")        
        visits_summary = pd.DataFrame({
            'day_obs': groups['day_obs'].first(),
            'seq': groups['seq'].median(),
            'psf_fwhm_05': groups['psf_fwhm'].quantile(0.05),
            'psf_fwhm_95': groups['psf_fwhm'].quantile(0.95),
            'psf_fwhm_sigma': groups['psf_fwhm'].std(),
        
            # NEW — ellipticity min/median/max (true across CCDs)
            'psf_ellipticity_median': groups['ellipticity'].median(),
            'psf_ellipticity_sigma': groups['ellipticity'].std(),
        })
        visits_summary['psf_fwhm_95_05'] = np.sqrt(visits_summary['psf_fwhm_95']**2 - visits_summary['psf_fwhm_05']**2)
        self.table = pd.merge(
                        self.table, visits_summary, how="left", on=["seq", "day_obs"])

        self.table = await find_faults(self.efd_client, self.table)
        
        if not simplified:
            unique_day_seq = (
                self.table[["day_obs", "seq", "obs_end", "obs_start"]]
                .drop_duplicates()
                .reset_index(drop=True)
            )
            (
                days,
                seqs,
                lut,
                cam_air_temp,
                states,
                m1m3_air_temp,
                outside_temp,
                m2_air_temp,
                correction_seq,
                ims_z
            ) = ([] for _ in range(10))
            for idx, row in tqdm(unique_day_seq.iterrows(), total=len(unique_day_seq), disable=True):
                day_obs = int(row["day_obs"])
                seq = int(row["seq"])

                rec_end = row["obs_end"]
                rec_start = row["obs_start"]

                # ---------- Environment variables -------------
                # ----------------------------------------------
                # Get state
                # M2 and M1M3 data may be added later, but for now we 
                # just fill those DOFs with nans.
                m2_dofs_lut = np.full(20, np.nan)
                m1m3_dofs_lut = np.full(20, np.nan)
                try:
                    cam_hexapod_data = getMostRecentRowWithDataBefore(
                        self.efd_client,
                        "lsst.sal.MTHexapod.logevent_compensationOffset",
                        Time(rec_end, scale="utc"),
                        maxSearchNMinutes=10,
                        where=lambda df: df["salIndex"] == 1,
                    )

                    m2_hexapod_data = getMostRecentRowWithDataBefore(
                        self.efd_client,
                        "lsst.sal.MTHexapod.logevent_compensationOffset",
                        Time(rec_end, scale="utc"),
                        maxSearchNMinutes=10,
                        where=lambda df: df["salIndex"] == 2,
                    )

                    hexapod_val = np.array(
                        [
                            m2_hexapod_data["z"],
                            m2_hexapod_data["x"],
                            m2_hexapod_data["y"],
                            m2_hexapod_data["u"],
                            m2_hexapod_data["v"],
                            cam_hexapod_data["z"],
                            cam_hexapod_data["x"],
                            cam_hexapod_data["y"],
                            cam_hexapod_data["u"],
                            cam_hexapod_data["v"],
                        ]
                    )
                    lut_val = np.concatenate([hexapod_val, m1m3_dofs_lut, m2_dofs_lut])
                except Exception:
                    lut_val = np.full(50, np.nan)

                event = getMostRecentRowWithDataBefore(
                    self.efd_client,
                    "lsst.sal.MTAOS.logevent_degreeOfFreedom",
                    timeToLookBefore=Time(rec_start, scale="utc"),
                )
                out = np.empty(
                    50,
                )
                for i in range(50):
                    out[i] = event[f"aggregatedDoF{i}"]
                states_val = out

                seq_num_corr = event["visitId"]
                
                # Get outside temperature
                outside_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="utc"),
                    Time(rec_end, scale="utc") + self.temp_time_window,
                    index=301,
                    convert_influx_index=True
                )
                if "temperatureItem0" in outside_temp_data:
                    outside_temp_val = outside_temp_data["temperatureItem0"].mean()
                else:
                    outside_temp_val = np.nan

                # Get M2 temperature
                m2_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="utc"),
                    Time(rec_end, scale="utc") + self.temp_time_window,
                    index=112,
                    convert_influx_index=True
                )
                if "temperatureItem0" in m2_temp_data:
                    m2_air_temp_val = m2_temp_data["temperatureItem0"].mean()
                else:
                    m2_air_temp_val = np.nan

                # Get cam temperature
                cam_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="utc"),
                    Time(rec_end, scale="utc") + self.temp_time_window,
                    index=111,
                )
                if "temperatureItem0" in cam_temp_data:
                    cam_air_temp_val = cam_temp_data["temperatureItem0"].mean()
                else:
                    cam_air_temp_val = np.nan

                # Get temperature above m1m3
                m1m3_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="utc"),
                    Time(rec_end, scale="utc") + self.temp_time_window,
                    index=113,
                    convert_influx_index=True
                )
                if "temperatureItem0" in m1m3_temp_data:
                    m1m3_air_temp_val = m1m3_temp_data["temperatureItem0"].mean()
                else:
                    m1m3_air_temp_val = np.nan

                # Get ims_z
                ims_z_data = await self.efd_client.select_time_series(
                    "lsst.sal.MTM1M3.imsData",
                    ["zPosition"],
                    Time(rec_start, scale="utc"),
                    Time(rec_end, scale="utc") + self.temp_time_window,
                    convert_influx_index=True,
                )
                if "zPosition" in ims_z_data:
                    ims_z_val = ims_z_data["zPosition"].median()
                else:
                    ims_z_val = np.nan

                lut.append(lut_val)
                m1m3_air_temp.append(m1m3_air_temp_val)
                cam_air_temp.append(cam_air_temp_val)
                m2_air_temp.append(m2_air_temp_val)
                outside_temp.append(outside_temp_val)
                states.append(states_val)
                days.append(day_obs)
                seqs.append(seq)
                correction_seq.append(seq_num_corr)
                ims_z.append(ims_z_val)

            # Calculate FWHM Zernike contributions
            efd_table = pd.DataFrame(
                {
                    "day_obs": days,
                    "seq": seqs,
                    "cam_air_temp": cam_air_temp,
                    "m2_air_temp": m2_air_temp,
                    "m1m3_air_temp": m1m3_air_temp,
                    "outside_temp": outside_temp,
                    "lut_state": lut,
                    "dof_state": states,
                    "seq_num_corr": correction_seq,
                    "ims_z": ims_z,
                }
            )

            self.table = pd.merge(
                self.table, efd_table, how="left", on=["seq", "day_obs"]
            )
            m1m3_gradient_table = await get_m1m3_gradients(self.efd_client, unique_day_seq)
            self.table = pd.merge(
                self.table, m1m3_gradient_table, how="left", on=["seq", "day_obs"]
            )
            self.table["m2_delta_t"] = (
                self.table["m2_air_temp"] - self.table["m1m3_air_temp"]
            )
            self.table["dome_delta_t"] = (
                self.table["outside_temp"] - self.table["m1m3_air_temp"]
            )
            self.table["cam_m1m3_delta_t"] = (
                self.table["cam_air_temp"] - self.table["m1m3_air_temp"]
            )

        return self.table


In [ ]:

zk_groups = [[0], [11 - 4], [1, 2], [3,4], [5, 6], [22]]
zk_group_labels = ['Z4', 'Z11', 'Z5 / Z6', 'Z7 / Z8', ' Z9 / Z10', 'Z22']

groups = [[0], [5], [1, 2], [6, 7], [3, 4], [8,9]]
group_labels = [' M2 dz', 'Cam dz', 'M2 dx/dy', 'Cam dx/dy', 'M2 tilts', 'Cam tilts']
labels = ['m2 dz', 'm2 dx', 'm2 dy', 'm2 rx', 'm2 ry',
          'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry']


mirror_groups = [[10, 11, 30, 31], [12, 34], [13, 14, 32, 33], [15, 16, 35, 36], [17, 18, 37, 38]]
mirror_group_labels = ['Astig', 'Spherical', 'Trefoil', 'Coma', 'Quad']
all_labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
     'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
     '$B_{{1,1}}$', '$B_{{1,2}}$', '$B_{{1,3}}$', '$B_{{1,4}}$', '$B_{{1,5}}$',
     '$B_{{1,6}}$', '$B_{{1,7}}$', '$B_{{1,8}}$', '$B_{{1,9}}$', '$B_{{1,10}}$',
     '$B_{{1,11}}$', '$B_{{1,12}}$', '$B_{{1,13}}$', '$B_{{1,14}}$', '$B_{{1,15}}$',
     '$B_{{1,16}}$', '$B_{{1,17}}$', '$B_{{1,18}}$', '$B_{{1,19}}$', '$B_{{1,20}}$',
     '$B_{{2,1}}$', '$B_{{2,2}}$', '$B_{{2,3}}$', '$B_{{2,4}}$', '$B_{{2,5}}$',
     '$B_{{2,6}}$', '$B_{{2,7}}$', '$B_{{2,8}}$', '$B_{{2,9}}$', '$B_{{2,10}}$',
     '$B_{{2,11}}$', '$B_{{2,12}}$', '$B_{{2,13}}$', '$B_{{2,14}}$', '$B_{{2,15}}$',
     '$B_{{2,16}}$', '$B_{{2,17}}$', '$B_{{2,18}}$', '$B_{{2,19}}$', '$B_{{2,20}}$'
     ]

## Simplified Nightly Report

## Full Nightly Report

In [ ]:
import os, sys 
os.environ["no_proxy"] += ",.consdb"
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
client = makeEfdClient()
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=False)

In [ ]:
day_obss = [20251101, 20251102, 20251104, 20251105, 20251107, 20251117, 20251121, 20251211, 20251223]
fig = plt.figure(figsize=(10,12))
plt.subplots_adjust(hspace=0.5)
z4_slopes = []
temp_slopes = []
ims_slopes = []
pdf = PdfPages(f"/home/c/cslage/u/MTAOS/data/Open_Loop_Drifts_21Jan26.pdf")                        
for day_obs in day_obss:
    axs = []
    for i in range(4):
        axs.append(fig.add_subplot(4,1,i+1))
    db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
    await db.create(simplified=False)
    table = db.table
    
    if len(table)>0:
        filtered_table = table[table['day_obs'] == day_obs]
        raw_filtered_table = table[table['day_obs'] == day_obs]
        filtered_table = filtered_table.select_dtypes(include="number")
        filtered_table['band'] = raw_filtered_table['band']
        filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
        
        # -- AOS jumps
        aos_diff = filtered_table["aos_fwhm"].diff()
        jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()
        
        # -- States array and labels
        states_per_seq = (
            raw_filtered_table[["seq", "dof_state", "zernikes_fwhm", "lut_state"]]
            .drop_duplicates("seq")
            .dropna(subset=["dof_state", "zernikes_fwhm", "lut_state"])
            .set_index("seq")
        )
        
        dof_state = np.vstack(states_per_seq["dof_state"].values)
        zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
        lut_state = np.vstack(states_per_seq["lut_state"].values)
        seqs = states_per_seq.index.values
    else:
        print(f'No  data available for {day_obs},  seq_nums {seq_min}-{seq_max}')

    blocks = find_blocks(table) # Find active blocks
    startSeq = blocks[blocks['block_after'] == 'T614']['seq'].values[0]
    endSeq = blocks[blocks['block_after'] == 'T614_triplets']['seq'].values[0] - 2
    if day_obs == 20251211:
        startSeq = 273
        endSeq = 322
    print(day_obs, startSeq, endSeq)
    open_seqs = list(range(startSeq, endSeq))
    these_seqs = filtered_table[filtered_table['seq'].isin(open_seqs)]
    z4s = these_seqs['z4'].values
    fit = np.polyfit(open_seqs, z4s, 1)
    z4_slope = fit[0]
    axs[0].scatter(open_seqs, z4s)
    ys = fit[1] + fit[0] * np.array(open_seqs)
    axs[0].plot(open_seqs, ys, ls='--', color='red')
    axs[0].set_title(f"BLOCK-T614 focus drift {day_obs}")
    axs[0].set_xlabel("Sequence number")
    axs[0].set_ylabel("Z4 (microns)")
    
    imszs = these_seqs['ims_z'].values * 1.0E6
    axs[1].scatter(open_seqs, imszs)
    axs[1].set_title(f"IMS_z values {day_obs}")
    axs[1].set_xlabel("Sequence number")
    axs[1].set_ylabel("IMS_z (microns)")
    #axs[1].set_ylim(-145, -130)

    expId1 = int(day_obs * 1E5 + startSeq)
    expId2 = int(day_obs * 1E5 + endSeq)
    dataId1 = {'exposure':expId1, 'instrument':'LSSTCam'}
    expRecord1 = getExpRecordFromDataId(butler, dataId1)
    dataId2 = {'exposure':expId2, 'instrument':'LSSTCam'}
    expRecord2 = getExpRecordFromDataId(butler, dataId2)
    tStart = expRecord1.timespan.begin.utc
    tEnd = expRecord2.timespan.end.utc

    ims_z = getEfdData(client,  "lsst.sal.MTM1M3.imsData",
        columns=["zPosition", "private_kafkaStamp"], begin=tStart, end=tEnd)
    times = ims_z['private_kafkaStamp'].values
    times -= times[0]
    z_vals = ims_z['zPosition'].values * 1.0E6
    fit = np.polyfit(times, z_vals, 1)
    ims_slope = fit[0]
    axs[2].plot(times, z_vals)
    xs = np.linspace(times[0], times[-1], 100)
    ys = fit[1] + xs * fit[0]
    axs[2].plot(xs, ys, ls='--', color='red')
    axs[2].set_title(f"BLOCK-T614 IMS Z drift {day_obs}")
    axs[2].set_xlabel("Time(seconds)")
    axs[2].set_ylabel("IMS z (microns)")
    truss_temp2 = getEfdData(client,  "lsst.sal.ESS.temperature",
        columns=["temperatureItem7", "salIndex", "private_kafkaStamp"], begin=tStart, end=tEnd)
    truss_temp2 = truss_temp2[truss_temp2['salIndex'] == 122]
    times = truss_temp2['private_kafkaStamp'].values
    times -= times[0]
    temps = truss_temp2['temperatureItem7'].values
    fit = np.polyfit(times, temps, 1)
    temp_slope = fit[0]
    axs[3].plot(times, temps)
    xs = np.linspace(times[0], times[-1], 100)
    ys = fit[1] + xs * fit[0]
    axs[3].plot(xs, ys, ls='--', color='red')
    axs[3].set_title(f"BLOCK-T614 temperature drift {day_obs}")
    axs[3].set_xlabel("Time(seconds)")
    axs[3].set_ylabel("Truss 2 Temperature (C)")
    z4_slope = z4_slope * (endSeq - startSeq) / (times[-1] - times[0])
    ims_slopes.append(ims_slope)
    z4_slopes.append(z4_slope)
    temp_slopes.append(temp_slope)
    pdf.savefig(fig)
    plt.clf()
pdf.close()    


In [ ]:
from lsst.summit.utils.efdUtils import calcNextDay
startDay = 20251206
endDay = 20251223
day_obs = startDay
fig = plt.figure(figsize=(10,12))
plt.subplots_adjust(hspace=0.5)
pdf = PdfPages(f"/home/c/cslage/u/MTAOS/data/Focus_Runs_21Jan26.pdf")                        
while day_obs < endDay:
    db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
    await db.create(simplified=False)
    table = db.table
    
    if len(table)>0:
        filtered_table = table[table['day_obs'] == day_obs]
        raw_filtered_table = table[table['day_obs'] == day_obs]
        filtered_table = filtered_table.select_dtypes(include="number")
        filtered_table['band'] = raw_filtered_table['band']
        filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
        
        # -- States array and labels
        states_per_seq = (
            raw_filtered_table[["seq", "dof_state", "zernikes_fwhm", "lut_state"]]
            .drop_duplicates("seq")
            .dropna(subset=["dof_state", "zernikes_fwhm", "lut_state"])
            .set_index("seq")
        )
        
        dof_state = np.vstack(states_per_seq["dof_state"].values)
        zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
        lut_state = np.vstack(states_per_seq["lut_state"].values)
        seqs = states_per_seq.index.values
    else:
        print(f'No  data available for {day_obs},  seq_nums {seq_min}-{seq_max}')
        day_obs = calcNextDay(day_obs)
        continue
    try:
        blocks = find_blocks(table) # Find active blocks
        print(day_obs, len(blocks))
        if len(blocks) == 0:
            print(f"{day_obs} has no blocks!")
            day_obs = calcNextDay(day_obs)
            continue
                
    
        for i in range(len(blocks)):
            if (blocks['block_after'].iloc[i] == 'T559_v2') or \
                (blocks['block_after'].iloc[i] == 'T559_v3'):
                try:
                    startSeq = blocks['seq'].iloc[i]
                    endSeq = blocks['seq'].iloc[i+1]
                    print(day_obs, startSeq, endSeq)
                except:
                    continue
                axs = []
                for i in range(4):
                    axs.append(fig.add_subplot(4,1,i+1))
    
                open_seqs = list(range(startSeq, endSeq))
                these_seqs = filtered_table[filtered_table['seq'].isin(open_seqs)]
                z4s = these_seqs['z4'].values
                fit = np.polyfit(open_seqs, z4s, 1)
                z4_slope = fit[0]
                axs[0].scatter(open_seqs, z4s)
                ys = fit[1] + fit[0] * np.array(open_seqs)
                axs[0].plot(open_seqs, ys, ls='--', color='red')
                axs[0].set_title(f"BLOCK-T559_v2 focus drift {day_obs}")
                axs[0].set_xlabel("Sequence number")
                axs[0].set_ylabel("Z4 (microns)")
                
                imszs = these_seqs['ims_z'].values * 1.0E6
                axs[1].scatter(open_seqs, imszs)
                axs[1].set_title(f"IMS_z values {day_obs}")
                axs[1].set_xlabel("Sequence number")
                axs[1].set_ylabel("IMS_z (microns)")
                #axs[1].set_ylim(-145, -130)
            
                expId1 = int(day_obs * 1E5 + startSeq)
                expId2 = int(day_obs * 1E5 + endSeq)
                dataId1 = {'exposure':expId1, 'instrument':'LSSTCam'}
                expRecord1 = getExpRecordFromDataId(butler, dataId1)
                dataId2 = {'exposure':expId2, 'instrument':'LSSTCam'}
                expRecord2 = getExpRecordFromDataId(butler, dataId2)
                tStart = expRecord1.timespan.begin.utc
                tEnd = expRecord2.timespan.end.utc
            
                ims_z = getEfdData(client,  "lsst.sal.MTM1M3.imsData",
                    columns=["zPosition", "private_kafkaStamp"], begin=tStart, end=tEnd)
                times = ims_z['private_kafkaStamp'].values
                times -= times[0]
                z_vals = ims_z['zPosition'].values * 1.0E6
                fit = np.polyfit(times, z_vals, 1)
                ims_slope = fit[0]
                axs[2].plot(times, z_vals)
                xs = np.linspace(times[0], times[-1], 100)
                ys = fit[1] + xs * fit[0]
                axs[2].plot(xs, ys, ls='--', color='red')
                axs[2].set_title(f"BLOCK-T559_v2 IMS Z drift {day_obs}")
                axs[2].set_xlabel("Time(seconds)")
                axs[2].set_ylabel("IMS z (microns)")
                truss_temp2 = getEfdData(client,  "lsst.sal.ESS.temperature",
                    columns=["temperatureItem7", "salIndex", "private_kafkaStamp"], begin=tStart, end=tEnd)
                truss_temp2 = truss_temp2[truss_temp2['salIndex'] == 122]
                times = truss_temp2['private_kafkaStamp'].values
                times -= times[0]
                temps = truss_temp2['temperatureItem7'].values
                fit = np.polyfit(times, temps, 1)
                temp_slope = fit[0]
                axs[3].plot(times, temps)
                xs = np.linspace(times[0], times[-1], 100)
                ys = fit[1] + xs * fit[0]
                axs[3].plot(xs, ys, ls='--', color='red')
                axs[3].set_title(f"BLOCK-T559_v2 temperature drift {day_obs}")
                axs[3].set_xlabel("Time(seconds)")
                axs[3].set_ylabel("Truss 2 Temperature (C)")
                pdf.savefig(fig)
                plt.clf()

    except:
        print(f"{day_obs} failed!")
        day_obs = calcNextDay(day_obs)
        continue
    print(f"{day_obs} succeeded!")
    day_obs = calcNextDay(day_obs)
pdf.close()    


In [ ]:
from lsst.summit.utils.efdUtils import calcNextDay
startDay = 20251206
endDay = 20251231
day_obs = startDay
while day_obs < endDay:
    db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
    await db.create(simplified=True)
    table = db.table
    try:
        blocks = find_blocks(table) # Find active blocks
        if len(blocks) == 0:
            print(f"{day_obs} has no blocks!")
            day_obs = calcNextDay(day_obs)
            continue
    
        for i in range(len(blocks)):
            if (blocks['block_after'].iloc[i] == 'T559_v2') or \
                (blocks['block_after'].iloc[i] == 'T559_v3'):
                startSeq = blocks['seq'].iloc[i]
                endSeq = blocks['seq'].iloc[i+1]
                print(day_obs, startSeq, endSeq)
        print(day_obs, len(startSeq))
    except:
        print(f"{day_obs} failed!")
        day_obs = calcNextDay(day_obs)
        continue
    day_obs = calcNextDay(day_obs)
    


In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,5))
plt.suptitle("Open loop BLOCK-T614 focus drifts", fontsize=18)
plt.subplots_adjust(wspace=0.3)
axs[0].scatter(z4_slopes, temp_slopes)
fit = np.polyfit(z4_slopes, temp_slopes,1)
xs = np.linspace(-0.0003, .0002, 100)
ys = fit[1] + fit[0] * np.array(xs)
axs[0].plot(xs, ys, ls='--', color='red')
axs[0].set_xlabel("Focus drift (microns/second)")
axs[0].set_ylabel("Truss2 Temp drift (degrees C/second)")
axs[1].scatter(z4_slopes, ims_slopes)
axs[1].set_xlabel("Focus drift (microns/second)")
axs[1].set_ylabel("IMS Z drift (mcrons/second)")
fit = np.polyfit(z4_slopes, ims_slopes,1)

ys = fit[1] + fit[0] * np.array(xs)
axs[1].plot(xs, ys, ls='--', color='red')
fig.savefig(f"/home/c/cslage/u/MTAOS/data/Open_Loop_Correlations_12Jan26.png")

In [ ]:
day_obss = [20251206]
fig = plt.figure(figsize=(10,10))
plt.subplots_adjust(hspace=0.5)
z4_slopes = []
temp_slopes = []
ims_slopes = []
pdf = PdfPages(f"/home/c/cslage/u/MTAOS/data/Open_Loop_Drifts_Focus_15Jan26.pdf")                        
for day_obs in day_obss:
    axs = []
    axs.append(fig.add_subplot(3,1,1))
    axs.append(fig.add_subplot(3,1,2))
    axs.append(fig.add_subplot(3,1,3))
    db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
    #await db.create(simplified=False)
    #table = db.table
    
    if len(table)>0:
        filtered_table = table[table['day_obs'] == day_obs]
        raw_filtered_table = table[table['day_obs'] == day_obs]
        filtered_table = filtered_table.select_dtypes(include="number")
        filtered_table['band'] = raw_filtered_table['band']
        filtered_table['time'] = raw_filtered_table['time']
        filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col in ['band', 'time'] else 'mean' for col in filtered_table.columns})
        
        #filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
        
        # -- AOS jumps
        aos_diff = filtered_table["aos_fwhm"].diff()
        jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()
        
        # -- States array and labels
        states_per_seq = (
            raw_filtered_table[["seq", "dof_state", "zernikes_fwhm", "lut_state"]]
            .drop_duplicates("seq")
            .dropna(subset=["dof_state", "zernikes_fwhm", "lut_state"])
            .set_index("seq")
        )
        
        dof_state = np.vstack(states_per_seq["dof_state"].values)
        zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
        lut_state = np.vstack(states_per_seq["lut_state"].values)
        seqs = states_per_seq.index.values
    else:
        print(f'No  data available for {day_obs},  seq_nums {seq_min}-{seq_max}')

    blocks = find_blocks(table) # Find active blocks
    startSeq = 40
    endSeq = 106
    print(day_obs, startSeq, endSeq)
    open_seqs = list(range(startSeq, endSeq))
    these_seqs = filtered_table[filtered_table['seq'].isin(open_seqs)]
    z4s = these_seqs['z4'].values
    fit = np.polyfit(open_seqs, z4s, 1)
    z4_slope = fit[0]
    axs[0].scatter(open_seqs, z4s)
    ys = fit[1] + fit[0] * np.array(open_seqs)
    #axs[0].plot(open_seqs, ys, ls='--', color='red')
    axs[0].set_title(f"BLOCK-T614 focus drift {day_obs}")
    axs[0].set_xlabel("Sequence number")
    axs[0].set_ylabel("Z4 (microns)")

    expId1 = int(day_obs * 1E5 + startSeq)
    expId2 = int(day_obs * 1E5 + endSeq)
    dataId1 = {'exposure':expId1, 'instrument':'LSSTCam'}
    expRecord1 = getExpRecordFromDataId(butler, dataId1)
    dataId2 = {'exposure':expId2, 'instrument':'LSSTCam'}
    expRecord2 = getExpRecordFromDataId(butler, dataId2)
    tStart = expRecord1.timespan.begin.utc
    tEnd = expRecord2.timespan.end.utc

    truss_temp2 = getEfdData(client,  "lsst.sal.ESS.temperature",
        columns=["temperatureItem7", "salIndex", "private_kafkaStamp"], begin=tStart, end=tEnd)
    truss_temp2 = truss_temp2[truss_temp2['salIndex'] == 122]
    times = truss_temp2['private_kafkaStamp'].values
    times -= times[0]
    temps = truss_temp2['temperatureItem7'].values
    fit = np.polyfit(times, temps, 1)
    temp_slope = fit[0]
    axs[1].plot(times, temps)
    xs = np.linspace(times[0], times[-1], 100)
    ys = fit[1] + xs * fit[0]
    axs[1].plot(xs, ys, ls='--', color='red')
    axs[1].set_title(f"BLOCK-T614 temperature drift {day_obs}")
    axs[1].set_xlabel("Time(seconds)")
    axs[1].set_ylabel("Temperature (C)")
    z4_slope = z4_slope * (endSeq - startSeq) / (times[-1] - times[0])
    ims_z = getEfdData(client,  "lsst.sal.MTM1M3.imsData",
        columns=["zPosition", "private_kafkaStamp"], begin=tStart, end=tEnd)
    times = ims_z['private_kafkaStamp'].values
    #times -= times[0]
    z_vals = ims_z['zPosition'].values * 1.0E6
    fit = np.polyfit(times, z_vals, 1)
    ims_slope = fit[0]
    axs[2].plot(times, z_vals)
    xs = np.linspace(times[0], times[-1], 100)
    ys = fit[1] + xs * fit[0]
    #axs[2].plot(xs, ys, ls='--', color='red')
    this_table = filtered_table[filtered_table['seq'] >= startSeq]
    this_table = this_table[filtered_table['seq'] <= endSeq]
    xticks = [Time(val, format='isot', scale='tai').unix_tai for val in this_table['time'].values]
    xticklabels = this_table['time'].index
    axs[2].set_xticks(xticks)
    axs[2].set_xticklabels(xticklabels)
    axs[2].set_title(f"BLOCK-T614 IMS Z drift {day_obs}")
    axs[2].set_xlabel("Time(seconds)")
    axs[2].set_ylabel("IMS z (microns)")
    ims_slopes.append(ims_slope)
    z4_slopes.append(z4_slope)
    temp_slopes.append(temp_slope)
    pdf.savefig(fig)
    plt.clf()
pdf.close()    


In [ ]:
db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
await db.create(simplified=False)
table = db.table
table.columns

In [ ]:
filtered_table['ims_z']

In [ ]:
day_obss = 20251206
fig = plt.figure(figsize=(10,10))
plt.subplots_adjust(hspace=0.3)
axs = []
axs.append(fig.add_subplot(2,1,1))
axs.append(fig.add_subplot(2,1,2))
#db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
#await db.create(simplified=False)
#table = db.table

if len(table)>0:
    filtered_table = table[table['day_obs'] == day_obs]
    raw_filtered_table = table[table['day_obs'] == day_obs]
    filtered_table = filtered_table.select_dtypes(include="number")
    filtered_table['band'] = raw_filtered_table['band']
    filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col in ['band'] else 'mean' for col in filtered_table.columns})
    
    #filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
    
    # -- AOS jumps
    aos_diff = filtered_table["aos_fwhm"].diff()
    jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()
    
    # -- States array and labels
    states_per_seq = (
        raw_filtered_table[["seq", "dof_state", "zernikes_fwhm", "lut_state"]]
        .drop_duplicates("seq")
        .dropna(subset=["dof_state", "zernikes_fwhm", "lut_state"])
        .set_index("seq")
    )
    
    dof_state = np.vstack(states_per_seq["dof_state"].values)
    zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
    lut_state = np.vstack(states_per_seq["lut_state"].values)
    seqs = states_per_seq.index.values
else:
    print(f'No  data available for {day_obs},  seq_nums {seq_min}-{seq_max}')

blocks = find_blocks(table) # Find active blocks
startSeq = 40
endSeq = 106
print(day_obs, startSeq, endSeq)
open_seqs = list(range(startSeq, endSeq))
these_seqs = filtered_table[filtered_table['seq'].isin(open_seqs)]
z4s = these_seqs['z4'].values
axs[0].scatter(open_seqs, z4s)
axs[0].set_title(f"Z4 values {day_obs}")
axs[0].set_xlabel("Sequence number")
axs[0].set_ylabel("Z4 (microns)")

imszs = these_seqs['ims_z'].values * 1.0E6
axs[1].scatter(open_seqs, imszs)
axs[1].set_title(f"IMS_z values {day_obs}")
axs[1].set_xlabel("Sequence number")
axs[1].set_ylabel("IMS_z (microns)")
axs[1].set_ylim(-145, -130)

seq_lines = [42, 67, 69, 75, 79, 81]
for ax in axs:
    for x in seq_lines:
        ax.axvline(x, ls='--', color='black')

fig.savefig(f"/home/c/cslage/u/MTAOS/data/Z4_IMS-z_Correlations_15Jan26.png")    